# Analise arquivos raw movimentacao-bancaria.csv

In [167]:
import os
import pandas as pd
from pandasql import sqldf

pysqldf = lambda q: sqldf(q, globals())

In [168]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath('.'))))
DATA_DIR = os.path.join(BASE_DIR, 'data')
RAW_DIR =  os.path.join(DATA_DIR, 'raw')

In [169]:
df = pd.read_csv(os.path.join(RAW_DIR, 'movimentacoes_bancarias.csv'), sep=';')
df.shape

(1015, 8)

## Analise exploratória

In [170]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1015 entries, 0 to 1014
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   id_movimento    1015 non-null   str  
 1   data_movimento  1015 non-null   str  
 2   tipo            994 non-null    str  
 3   categoria       995 non-null    str  
 4   descricao       995 non-null    str  
 5   valor           995 non-null    str  
 6   conta_bancaria  995 non-null    str  
 7   conciliado      995 non-null    str  
dtypes: str(8)
memory usage: 63.6 KB


In [171]:
df.isna().sum()

id_movimento       0
data_movimento     0
tipo              21
categoria         20
descricao         20
valor             20
conta_bancaria    20
conciliado        20
dtype: int64

In [172]:
df.head()

,id_movimento,data_movimento,tipo,categoria,descricao,valor,conta_bancaria,conciliado
0,MB000001,2026-05-14,Entrada,Outros,Outros - operação 1,11974.55,Banco B,Sim
1,MB000002,2026-03-15,Entrada,Energia,Energia - operação 2,329.88,Banco A,Sim
2,MB000003,2026-03-14,Saída,Pagamento fornecedor,NaN,1004.82,Banco B,Sim
3,MB000004,2026-04-21,Saída,Recebimento de cliente,Recebimento de cliente - operação 4,945.74,Banco A,Sim
4,MB000005,2026-08-13,Entrada,Outros,Outros - operação 5,11992.6,Banco C,Sim


In [173]:
df.tail()

,id_movimento,data_movimento,tipo,categoria,descricao,valor,conta_bancaria,conciliado
1010,MB000025,2026-06-04,Entrada,Outros,Outros - operação 25,1788.7,Banco A,Sim
1011,MB000243,2026-03-21,Entrada,Outros,Outros - operação 243,6344.79,Banco B,Sim
1012,MB000543,2026-06-12,Saída,Transferência,Transferência - operação 543,3313.07,Banco A,Sim
1013,MB000931,2026-07-29,Entrada,Energia,Energia - operação 931,7136.89,Banco A,sim
1014,MB000887,2026-03-18,Saída,Tarifa bancária,Tarifa bancária - operação 887,970.59,Banco C,Sim


In [174]:
df.sample(5)

,id_movimento,data_movimento,tipo,categoria,descricao,valor,conta_bancaria,conciliado
382,MB000383,2026-02-10,Entrada,Tarifa bancária,Tarifa bancária - operação 383,14155.31,Banco A,Não
243,MB000244,2026-05-26,Entrada,NaN,Transferência - operação 244,1632.51,Banco A,Sim
710,MB000711,2026-04-11,Entrada,Energia,Energia - operação 711,8157.11,Banco B,Não
934,MB000935,2026-07-16,Saída,Recebimento de cliente,Recebimento de cliente - operação 935,8964.61,Banco C,Sim
101,MB000102,2026-02-16,Entrada,Energia,Energia - operação 102,4708.76,Banco B,Sim


In [175]:
df.isnull().sum()

id_movimento       0
data_movimento     0
tipo              21
categoria         20
descricao         20
valor             20
conta_bancaria    20
conciliado        20
dtype: int64

In [176]:
q = '''SELECT id_movimento
            , tipo
            , categoria
            , descricao
            , valor
            , conta_bancaria
            , conciliado
        FROM df
        WHERE tipo is null
            or categoria isnull
            or valor isnull
            or conta_bancaria isnull
            or conciliado isnull

'''
pysqldf(q)

,id_movimento,tipo,categoria,descricao,valor,conta_bancaria,conciliado
0,MB000027,Saída,Transferência,Transferência - operação 27,17689.4,NaN,Sim
1,MB000032,Entrada,Transferência,Transferência - operação 32,NaN,Banco B,Sim
2,MB000040,NaN,Salário,Salário - operação 40,10389.9,Banco C,Sim
3,MB000046,Entrada,NaN,Transferência - operação 46,13752.17,Banco C,Sim
4,MB000063,NaN,Aluguel,Aluguel - operação 63,8909.31,Banco C,Não
...,...,...,...,...,...,...,...
95,MB000939,Entrada,NaN,Salário - operação 939,16762.92,Banco C,Sim
96,MB000945,Entrada,Transferência,Transferência - operação 945,NaN,Banco B,Sim
97,MB000991,Saída,Energia,Energia - operação 991,7902.24,Banco B,NaN
98,MB000996,Entrada,Transferência,Transferência - operação 996,3656.59,Banco B,NaN


## Tratamento de dados

In [177]:
df_original = df.copy()

In [178]:
df = df.fillna('NAO INFORMADO')

In [179]:
# Remove espaços antes/depois dos nomes
df.columns = df.columns.str.strip()

In [180]:
campos_texto = [
    "id_movimento"
    , "tipo"
    , "categoria"
    , "descricao"
    , "conta_bancaria"
    , "conciliado"
]

In [181]:
for campo in campos_texto:

    df[campo] = (
        df[campo]
        .astype("string")
        .str.strip()
    )

In [182]:
# campos datas

df['data_movimento'] = df['data_movimento'].replace('NAO INFORMADO', '1900-01-01', regex=True)
df['data_movimento'] = df['data_movimento'].replace('/', '-',  regex=True)

In [183]:
df['data_movimento'] = pd.to_datetime(df['data_movimento'], format='mixed')

In [184]:
df['valor'] = df['valor'].replace('NAO INFORMADO', 0)

In [187]:
df['valor'] = pd.to_numeric(df['valor'], errors='coerce')

In [188]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1015 entries, 0 to 1014
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_movimento    1015 non-null   string        
 1   data_movimento  1015 non-null   datetime64[us]
 2   tipo            1015 non-null   string        
 3   categoria       1015 non-null   string        
 4   descricao       1015 non-null   string        
 5   valor           1014 non-null   float64       
 6   conta_bancaria  1015 non-null   string        
 7   conciliado      1015 non-null   string        
dtypes: datetime64[us](1), float64(1), string(6)
memory usage: 63.6 KB


In [191]:
df.head()

,id_movimento,data_movimento,tipo,categoria,descricao,valor,conta_bancaria,conciliado
0,MB000001,2026-05-14,Entrada,Outros,Outros - operação 1,11974.55,Banco B,Sim
1,MB000002,2026-03-15,Entrada,Energia,Energia - operação 2,329.88,Banco A,Sim
2,MB000003,2026-03-14,Saída,Pagamento fornecedor,NAO INFORMADO,1004.82,Banco B,Sim
3,MB000004,2026-04-21,Saída,Recebimento de cliente,Recebimento de cliente - operação 4,945.74,Banco A,Sim
4,MB000005,2026-08-13,Entrada,Outros,Outros - operação 5,11992.60,Banco C,Sim


In [192]:
df.tail()

,id_movimento,data_movimento,tipo,categoria,descricao,valor,conta_bancaria,conciliado
1010,MB000025,2026-06-04,Entrada,Outros,Outros - operação 25,1788.70,Banco A,Sim
1011,MB000243,2026-03-21,Entrada,Outros,Outros - operação 243,6344.79,Banco B,Sim
1012,MB000543,2026-06-12,Saída,Transferência,Transferência - operação 543,3313.07,Banco A,Sim
1013,MB000931,2026-07-29,Entrada,Energia,Energia - operação 931,7136.89,Banco A,sim
1014,MB000887,2026-03-18,Saída,Tarifa bancária,Tarifa bancária - operação 887,970.59,Banco C,Sim


In [193]:
df.sample(5)

,id_movimento,data_movimento,tipo,categoria,descricao,valor,conta_bancaria,conciliado
930,MB000931,2026-07-29,Entrada,Energia,Energia - operação 931,7136.89,Banco A,sim
129,MB000130,2026-06-26,Entrada,Pagamento fornecedor,Pagamento fornecedor - operação 130,2426.18,Banco C,Sim
135,MB000136,2026-07-07,Entrada,Outros,Outros - operação 136,1554.02,Banco A,Não
261,MB000262,2026-04-30,Saída,Salário,Salário - operação 262,13549.21,Banco B,Não
790,MB000791,2026-06-07,Saída,NAO INFORMADO,Tarifa bancária - operação 791,11646.44,Banco C,Sim


## Salvando dados Movimentacoes Bancarias em Bronze

In [194]:
BRONZE_DIR = os.path.join(DATA_DIR, 'bronze')

In [195]:
df.to_csv(
    os.path.join(BRONZE_DIR, 'b_movimentacoes_bancarias.csv')
    , index=False
    , date_format="%Y-%m-%d"
)